# Clean Docling Hybrid RAG — reference implementation

Cleaned from `RAG_Quad (1).ipynb`. This notebook is the **reference logic for Codex**, not the final FastAPI application. It removes exploratory/duplicate cells, uses local paths and environment variables instead of Google Colab, and retains the working retrieval approach.

**Pipeline:** Docling PDF conversion → separate text/table documents → split text (tables kept whole) → OpenAI embeddings → Qdrant semantic retrieval + BM25 lexical retrieval → 50/50 ensemble → GPT-4.1-mini answer.

**Limitations inherited from the original:** Qdrant is in-memory; BM25 is rebuilt on each run; individual text items may create tiny chunks; large tables may exceed ideal retrieval sizes; reliable PDF page citations are not yet implemented. The sample prompt asks for page citations *only when available*; do not fabricate page numbers.

## 1. Install dependencies
Run once in your Python environment. For VS Code, prefer a virtual environment and install dependencies via the terminal. The versions should be locked after the first successful local test.

In [ ]:
%pip install -U docling langchain-classic langchain-community langchain-core langchain-openai langchain-text-splitters langchain-qdrant qdrant-client rank_bm25 python-dotenv

## 2. Configuration
Create a local `.env` with `OPENAI_API_KEY=...` (never commit it). Put a PDF at `data/demo_clinic.pdf`, or change `PDF_PATH` to an existing local file. The original Colab path was `/content/Ragfile`.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY missing. Add it to an uncommitted .env file.")

PDF_PATH = Path("data/demo_clinic.pdf")  # replace with your local PDF path
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"Place a PDF here or update PDF_PATH: {PDF_PATH.resolve()}")

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 3. Docling extraction

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
conversion_result = converter.convert(str(PDF_PATH))
docling_document = conversion_result.document
print("Tables detected:", len(docling_document.tables))

## 4. Text/table-aware chunking
Preserves the original approach: text elements split at 1,500 characters with 200 overlap; Markdown tables remain whole. Metadata includes source path and document type, **not verified page numbers**.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from docling_core.types.doc import TableItem

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)
text_docs, table_docs = [], []
for item, _level in docling_document.iterate_items():
    if isinstance(item, TableItem):
        table_docs.append(Document(
            page_content=item.export_to_markdown(doc=docling_document),
            metadata={"source": str(PDF_PATH), "type": "table"},
        ))
    elif hasattr(item, "text") and isinstance(item.text, str) and item.text.strip():
        text_docs.append(Document(
            page_content=item.text,
            metadata={"source": str(PDF_PATH), "type": "text"},
        ))

text_chunks = text_splitter.split_documents(text_docs)
chunks = text_chunks + table_docs
if not chunks:
    raise RuntimeError("Docling produced no searchable chunks.")
print(f"Text chunks: {len(text_chunks)} | Tables: {len(table_docs)} | Total: {len(chunks)}")

## 5. Build Qdrant semantic and BM25 lexical retrievers
**Demo only:** `location=":memory:"` is not persistent. For the FastAPI app, use a persistent Qdrant instance and retain/rebuild the BM25 corpus.

In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="docling_pdf_collection",
)
semantic_retriever = vector_store.as_retriever(search_kwargs={"k": 5})
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5
hybrid_retriever = EnsembleRetriever(
    retrievers=[semantic_retriever, bm25_retriever],
    weights=[0.5, 0.5],
)
print("Hybrid retriever ready")

## 6. Prompt and RAG chain
Uses the final `QA_CHAIN_PROMPT` from the uploaded notebook, not the earlier experimental `PROMPT`.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an AI assistant answering questions from a PDF document.

Use only the provided context to answer the question.

Rules:
1. Do not invent information.
2. If the answer is not available in the context, say:
   "I could not find this information in the document."
3. Read tables carefully and preserve numerical values and units.
4. If calculations are required, show the formula and steps.
5. Compare information across different sections when necessary.
6. Cite PDF page numbers when available in the context.
7. Give clear and accurate answers.

Context:
{context}

Question:
{question}

Answer:
""",
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=hybrid_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
)
print("Docling Hybrid RAG ready")

## 7. One smoke test
For the original clinical protocol PDF, the expected answer is **Screening, Visit 1 (Vaccination 1), Visit 4 (Vaccination 2)**. For any other PDF, replace the question and check against its contents. One correct answer is not a complete evaluation.

In [ ]:
question = (
    "According to the Phase 1 Schedule of Activities, "
    "at which visits is a urine pregnancy test performed (if appropriate)?"
)
response = qa_chain.invoke({"query": question})
print("ANSWER:\n", response["result"])
print("\nRETRIEVED SOURCES:")
for i, doc in enumerate(response["source_documents"], 1):
    print(f"{i}. {doc.metadata.get('source')} | {doc.metadata.get('type')} | {len(doc.page_content)} chars")

## Codex handoff

See `CODEX_INSTRUCTIONS.md` for the requested FastAPI + React architecture, behavior to preserve, known issues, API suggestions, and acceptance criteria. The notebook does **not** include a working 80-PDF index: the ZIP upload/selection cells in the source were exploratory and are intentionally excluded.